# Telecom Customer Churn — EDA & Threshold Tuning

Explores key churn segments in `data/telecom_churn_dataset.csv` and evaluates precision, recall, and F1 across probability thresholds using `models/churn_pipeline.joblib`.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
from sklearn.model_selection import train_test_split

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = project_root / 'data' / 'telecom_churn_dataset.csv'
pipeline_path = project_root / 'models' / 'churn_pipeline.joblib'

df = pd.read_csv(data_path)
df.columns = [c.strip().lower() for c in df.columns]
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
df['churn_flag'] = df['churn'].map({'No': 0, 'Yes': 1})

print('Shape:', df.shape)
print('Overall churn rate:', f"{df['churn_flag'].mean():.2%}")
print('Blank totalcharges rows:', int(df['totalcharges'].isna().sum()))

Shape: (7043, 22)
Overall churn rate: 26.54%
Blank totalcharges rows: 11


## Churn Rate by Key Segments

Check contract type, internet service, and tenure bands to see where churn concentrates.

In [2]:
for col in ['contract', 'internetservice', 'onlinesecurity', 'techsupport']:
    summary = (
        df.groupby(col, observed=False)['churn_flag']
        .agg(customers='count', churn_rate='mean')
        .assign(churn_rate=lambda s: (s['churn_rate'] * 100).round(2))
        .sort_values('churn_rate', ascending=False)
    )
    print(f'\n--- {col} ---')
    print(summary)

df['tenure_band'] = pd.cut(
    df['tenure'],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=['0-6m', '7-12m', '13-24m', '25-48m', '49-72m']
)
tenure_summary = (
    df.groupby('tenure_band', observed=False)['churn_flag']
    .agg(customers='count', churn_rate='mean')
    .assign(churn_rate=lambda s: (s['churn_rate'] * 100).round(2))
)
print('\n--- tenure_band ---')
print(tenure_summary)


--- contract ---
                customers  churn_rate
contract                             
Month-to-month       3875       42.71
One year             1473       11.27
Two year             1695        2.83

--- internetservice ---
                 customers  churn_rate
internetservice                       
Fiber optic           3096       41.89
DSL                   2421       18.96
No                    1526        7.40

--- onlinesecurity ---
                     customers  churn_rate
onlinesecurity                            
No                        3498       41.77
Yes                       2019       14.61
No internet service       1526        7.40

--- techsupport ---
                     customers  churn_rate
techsupport                               
No                        3473       41.64
Yes                       2044       15.17
No internet service       1526        7.40

--- tenure_band ---
             customers  churn_rate
tenure_band                       
0-6m  

## Probability Threshold Evaluation

Reconstruct the same stratified 80/20 split (`random_state=42`) used in `src/train.py`, score `X_test`, and compare thresholds.

In [3]:
X = df.drop(columns=[c for c in ['customerid', 'churn', 'churn_flag', 'tenure_band'] if c in df.columns])
y = df['churn_flag']

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = joblib.load(pipeline_path)
y_proba = pipeline.predict_proba(X_test)[:, 1]

rows = []
for th in [0.35, 0.45, 0.50, 0.55, 0.60, 0.65]:
    preds = (y_proba >= th).astype(int)
    tp = int(((preds == 1) & (y_test == 1)).sum())
    fp = int(((preds == 1) & (y_test == 0)).sum())
    fn = int(((preds == 0) & (y_test == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = f1_score(y_test, preds)
    rows.append({
        'threshold': th,
        'flagged_churn': int(preds.sum()),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4),
    })

thresh_df = pd.DataFrame(rows)
best_row = thresh_df.loc[thresh_df['f1'].idxmax()]
print(thresh_df.to_string(index=False))
print(f"\nHighest F1 in grid at threshold={best_row['threshold']} (F1={best_row['f1']})")

best_th = float(best_row['threshold'])
print(f'\nClassification Report @ threshold={best_th}:')
print(classification_report(y_test, (y_proba >= best_th).astype(int), target_names=['No Churn', 'Churn']))

 threshold  flagged_churn  precision  recall     f1
      0.35            681     0.4714  0.8583 0.6085
      0.45            585     0.5111  0.7995 0.6236
      0.50            543     0.5322  0.7727 0.6303
      0.55            504     0.5476  0.7380 0.6287
      0.60            455     0.5604  0.6818 0.6152
      0.65            392     0.5944  0.6230 0.6084

Highest F1 in grid at threshold=0.5 (F1=0.6303)

Classification Report @ threshold=0.5:
              precision    recall  f1-score   support

    No Churn       0.90      0.75      0.82      1035
       Churn       0.53      0.77      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409

